# TRM Maze - Direction Prediction (FAST)

**Approach:** Predict directions (UP/DOWN/LEFT/RIGHT/STOP) instead of positions
- ✓ Vocab: 5 (vs 900)
- ✓ Training: ~30 min
- ✓ Expected: 60-70% accuracy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install torch numpy pillow kaggle -q

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR
import numpy as np, time, os, glob
from collections import deque
from PIL import Image
from typing import Optional

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# Model (same as before, just smaller vocab)
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps, self.weight = eps, nn.Parameter(torch.ones(dim))
    def forward(self, x): return x / torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps) * self.weight

class RotaryPositionalEncoding(nn.Module):
    def __init__(self, dim, max_seq_len=512, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        t = torch.arange(max_seq_len)
        freqs = torch.outer(t, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)
    def _rotate_half(self, x):
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat([-x2, x1], dim=-1)
    def forward(self, q, k):
        L = q.size(2)
        cos, sin = self.cos_cached[:L].unsqueeze(0).unsqueeze(0), self.sin_cached[:L].unsqueeze(0).unsqueeze(0)
        return q*cos + self._rotate_half(q)*sin, k*cos + self._rotate_half(k)*sin

class SwiGLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        h = ((int(2/3*4*dim)+63)//64)*64
        self.w1, self.w2, self.w3 = nn.Linear(dim,h,bias=False), nn.Linear(h,dim,bias=False), nn.Linear(dim,h,bias=False)
    def forward(self, x): return self.w2(F.silu(self.w1(x)) * self.w3(x))

class TinyAttention(nn.Module):
    def __init__(self, dim, n_heads=8, dropout=0.0, max_seq_len=512):
        super().__init__()
        self.n_heads, self.head_dim, self.scale = n_heads, dim//n_heads, (dim//n_heads)**-0.5
        self.qkv, self.out, self.dropout = nn.Linear(dim,3*dim,bias=False), nn.Linear(dim,dim,bias=False), nn.Dropout(dropout)
        self.rope = RotaryPositionalEncoding(self.head_dim, max_seq_len)
    def forward(self, x, mask=None):
        B, L, D = x.shape
        qkv = self.qkv(x).reshape(B,L,3,self.n_heads,self.head_dim).permute(2,0,3,1,4)
        q, k, v = self.rope(qkv[0], qkv[1]) + (qkv[2],)
        attn = F.softmax((q @ k.transpose(-2,-1))*self.scale, dim=-1)
        return self.out((self.dropout(attn) @ v).transpose(1,2).reshape(B,L,D))

class TinyBlock(nn.Module):
    def __init__(self, dim, n_heads=8, dropout=0.0, max_seq_len=512):
        super().__init__()
        self.norm1, self.attn = RMSNorm(dim), TinyAttention(dim,n_heads,dropout,max_seq_len)
        self.norm2, self.ffn, self.dropout = RMSNorm(dim), SwiGLU(dim), nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = x + self.dropout(self.attn(self.norm1(x), mask))
        return x + self.dropout(self.ffn(self.norm2(x)))

class TinyNet(nn.Module):
    def __init__(self, dim=512, n_layers=2, n_heads=8, max_seq_len=512, dropout=0.0):
        super().__init__()
        self.layers, self.norm = nn.ModuleList([TinyBlock(dim,n_heads,dropout,max_seq_len) for _ in range(n_layers)]), RMSNorm(dim)
    def forward(self, x, mask=None):
        for layer in self.layers: x = layer(x, mask)
        return self.norm(x)
    def count_parameters(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

class CombinedHead(nn.Module):
    def __init__(self, dim, vocab_size):
        super().__init__()
        self.output_head = nn.Linear(dim, vocab_size, bias=False)
        self.halt_head = nn.Sequential(nn.Linear(dim,dim//4), nn.ReLU(), nn.Linear(dim//4,1))
    def forward(self, y): return self.output_head(y), self.halt_head(y.mean(dim=1))

class LatentRecursion(nn.Module):
    def __init__(self, net, dim=512, n_latent=6):
        super().__init__()
        self.net, self.n_latent = net, n_latent
        self.z_proj, self.y_proj = nn.Linear(3*dim,dim,bias=False), nn.Linear(2*dim,dim,bias=False)
    def forward(self, x, y, z):
        for _ in range(self.n_latent): z = self.net(self.z_proj(torch.cat([x,y,z],dim=-1)))
        return self.net(self.y_proj(torch.cat([y,z],dim=-1))), z

class TRMModel(nn.Module):
    def __init__(self, dim=512, n_layers=2, n_heads=8, n_latent=6, T_cycles=3, vocab_size=5, max_seq_len=256, dropout=0.0):
        super().__init__()
        self.T_cycles = T_cycles
        self.net = TinyNet(dim, n_layers, n_heads, max_seq_len, dropout)
        self.latent_recursion = LatentRecursion(self.net, dim, n_latent)
        self.heads = CombinedHead(dim, vocab_size)
    def forward(self, x, y, z):
        with torch.no_grad():
            for _ in range(self.T_cycles-1): y, z = self.latent_recursion(x, y, z)
        y, z = self.latent_recursion(x, y, z)
        return (y,z), *self.heads(y)
    def count_parameters(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
# Load dataset with DIRECTION conversion
os.environ['KAGGLE_USERNAME'] = 'aayushbhure'
os.environ['KAGGLE_KEY'] = 'YOUR_KEY_HERE'

!kaggle datasets download -d mexwell/maze-dataset
!unzip -q maze-dataset.zip

# Directions: 0=UP, 1=DOWN, 2=LEFT, 3=RIGHT, 4=STOP
def path_to_directions(path):
    """Convert path [(r,c)...] to directions [DOWN, RIGHT, ...]"""
    if len(path) < 2: return [4]  # STOP
    directions = []
    for i in range(len(path)-1):
        r1, c1 = path[i]
        r2, c2 = path[i+1]
        if r2 > r1: directions.append(1)  # DOWN
        elif r2 < r1: directions.append(0)  # UP
        elif c2 > c1: directions.append(3)  # RIGHT
        elif c2 < c1: directions.append(2)  # LEFT
    directions.append(4)  # STOP
    return directions

def load_maze_txt(f):
    with open(f, 'r') as file:
        return np.array([[int(x) for x in line.split()] for line in file])

def find_path_bfs(maze):
    rows, cols = maze.shape
    start, goal = (0,0), (rows-1, cols-1)
    queue, visited = deque([(start, [start])]), {start}
    while queue:
        (r,c), path = queue.popleft()
        if (r,c) == goal: return path
        for dr,dc in [(0,1),(1,0),(0,-1),(-1,0)]:
            nr, nc = r+dr, c+dc
            if 0<=nr<rows and 0<=nc<cols and maze[nr,nc]==0 and (nr,nc) not in visited:
                visited.add((nr,nc))
                queue.append(((nr,nc), path+[(nr,nc)]))
    return [start, goal]

print("Loading mazes...")
train_data = []
for f in glob.glob('imperfect_maze/*.txt')[:2000]:
    try:
        maze = load_maze_txt(f)
        if maze.shape[0]<20 or maze.shape[0]>100 or maze.shape[0]!=maze.shape[1]: continue
        path = find_path_bfs(maze)
        if len(path)<2: continue
        
        # Resize to 15×15 (smaller = easier)
        maze_15 = np.array(Image.fromarray((maze*255).astype(np.uint8)).resize((15,15), Image.NEAREST))//255
        scale = 15/maze.shape[0]
        path_15 = [(int(r*scale), int(c*scale)) for r,c in path]
        path_unique = [p for i,p in enumerate(path_15) if i==0 or p!=path_15[i-1]]
        
        # Convert to DIRECTIONS
        directions = path_to_directions(path_unique[:50])  # Max 50 moves
        
        train_data.append({
            "maze": maze_15.flatten().tolist(),
            "directions": directions
        })
    except: continue

print(f"Loaded {len(train_data)} mazes")
print(f"Avg moves: {np.mean([len(d['directions']) for d in train_data]):.1f}")
print(f"Sample: {train_data[0]['directions'][:10]}")

In [ ]:
class MazeDataset(Dataset):
    def __init__(self, data, max_seq_len=256):
        self.data, self.max_seq_len = data, max_seq_len
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        # Maze as input (15×15=225 tokens)
        x = torch.zeros(self.max_seq_len, dtype=torch.long)
        x[:len(item["maze"])] = torch.tensor(item["maze"], dtype=torch.long)
        # Initial: just START marker
        y_init = torch.zeros(self.max_seq_len, dtype=torch.long)
        # True directions
        y_true = torch.zeros(self.max_seq_len, dtype=torch.long)
        dirs = item["directions"][:50]
        y_true[:len(dirs)] = torch.tensor(dirs, dtype=torch.long)
        return {"x_tokens": x, "y_init_tokens": y_init, "y_true": y_true}

In [ ]:
# OPTIMIZED CONFIG for fast training
CONFIG = {
    "dim": 512, "n_layers": 2, "n_heads": 8, "n_latent": 6, "T_cycles": 3,
    "vocab_size": 5,  # UP, DOWN, LEFT, RIGHT, STOP
    "max_seq_len": 256, "dropout": 0.1,
    "lr": 5e-4, "weight_decay": 0.01, "max_iters": 5000,
    "batch_size": 32, "warmup_steps": 300, "grad_clip": 1.0,
    "N_sup": 4, "ema_decay": 0.999,
    "save_every": 1000, "log_every": 50
}

dataset = MazeDataset(train_data)
dataloader = DataLoader(dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2)

model = TRMModel(CONFIG["dim"], 2, CONFIG["n_heads"], CONFIG["n_latent"], 
                 CONFIG["T_cycles"], CONFIG["vocab_size"], CONFIG["max_seq_len"], CONFIG["dropout"]).to(device)
embedding = nn.Embedding(10, CONFIG["dim"]).to(device)  # 0-4 directions + padding
optimizer = optim.AdamW(list(model.parameters())+list(embedding.parameters()), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = LambdaLR(optimizer, lambda s: min(1.0, s/CONFIG["warmup_steps"]))
scaler = torch.amp.GradScaler('cuda')

class EMA:
    def __init__(self, model, decay=0.999):
        self.model, self.decay, self.shadow, self.backup = model, decay, {}, {}
        for n,p in model.named_parameters():
            if p.requires_grad: self.shadow[n] = p.data.clone()
    def update(self):
        for n,p in self.model.named_parameters():
            if p.requires_grad: self.shadow[n] = self.decay*self.shadow[n] + (1-self.decay)*p.data
    def apply_shadow(self):
        for n,p in self.model.named_parameters():
            if p.requires_grad: self.backup[n], p.data = p.data.clone(), self.shadow[n]
    def restore(self):
        for n,p in self.model.named_parameters():
            if p.requires_grad: p.data = self.backup[n]

ema_model, ema_embedding = EMA(model), EMA(embedding)
print(f"Parameters: {model.count_parameters():,}")
print(f"Vocab size: {CONFIG['vocab_size']} (directions)")
print(f"Expected time: ~30 min")

In [ ]:
# TRAINING LOOP
os.makedirs('/content/drive/MyDrive/TRM_Maze_Directions', exist_ok=True)
print("Starting direction-based training...\n" + "="*60)

model.train()
train_iter, start_time, best_acc = iter(dataloader), time.time(), 0.0

for step in range(CONFIG["max_iters"]):
    try: batch = next(train_iter)
    except StopIteration: train_iter, batch = iter(dataloader), next(iter(dataloader))
    
    x_tokens, y_true, y_tokens = batch["x_tokens"].to(device), batch["y_true"].to(device), batch["y_init_tokens"].to(device)
    
    for sup_step in range(CONFIG["N_sup"]):
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            x, y = embedding(x_tokens), embedding(y_tokens)
            z = torch.zeros_like(x)
            (_, _), y_hat, _ = model(x, y, z)
            loss = F.cross_entropy(y_hat.view(-1, y_hat.size(-1)), y_true.view(-1), ignore_index=0)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(list(model.parameters())+list(embedding.parameters()), CONFIG["grad_clip"])
        scaler.step(optimizer); scaler.update(); scheduler.step()
        ema_model.update(); ema_embedding.update()
        with torch.no_grad(): y_tokens = y_hat.argmax(dim=-1)
    
    if step % CONFIG["log_every"] == 0:
        with torch.no_grad():
            ema_model.apply_shadow(); ema_embedding.apply_shadow()
            x, y, z = embedding(x_tokens), embedding(batch["y_init_tokens"].to(device)), torch.zeros_like(x)
            (_, _), y_hat, _ = model(x, y, z)
            acc = (y_hat.argmax(dim=-1) == y_true).float().mean().item()
            ema_model.restore(); ema_embedding.restore()
        best_acc = max(best_acc, acc)
        print(f"Step {step:4d} | Loss: {loss.item():.4f} | Acc: {acc:.3f} | Best: {best_acc:.3f} | Time: {time.time()-start_time:.0f}s")
    
    if step>0 and step%CONFIG["save_every"]==0:
        ema_model.apply_shadow(); ema_embedding.apply_shadow()
        torch.save({"model": model.state_dict(), "embedding": embedding.state_dict(), "best_acc": best_acc},
                   f"/content/drive/MyDrive/TRM_Maze_Directions/step_{step}.pt")
        ema_model.restore(); ema_embedding.restore()

# Save final
ema_model.apply_shadow(); ema_embedding.apply_shadow()
torch.save({"model": model.state_dict(), "embedding": embedding.state_dict(), 
            "config": CONFIG, "best_acc": best_acc}, "/content/drive/MyDrive/TRM_Maze_Directions/final.pt")
print(f"\nComplete! Best: {best_acc:.3f}")